**Daily Weather Time Series Data - Delhi, India**

https://www.kaggle.com/datasets/yug201/daily-climate-time-series-data-delhi-india?select=kaggel_weather_2013_to_2024.csv

**Тема:** Прогнозування часових рядів на основі кліматичних даних

**Мета роботи:**
Навчитися виконувати аналіз часових рядів, проводити дослідницький аналіз даних (EDA), визначати автокореляційні залежності, будувати моделі прогнозування та оцінювати їхню точність.

**Вихідні дані**

Для виконання роботи використовується датасет Daily Climate Time Series Data (Delhi, India) з платформи Kaggle, який містить щоденні кліматичні показники (температура, вологість, опади тощо) за кілька років.

**Виконав:** Харченко Артьом, група ФІТ 3-10


1. Завантажити датасет з Kaggle.

2. Імпортувати дані у середовище Python (Jupyter Notebook або Google Colab).

3. Ознайомитися зі структурою датасету (head(), info(), describe()).

4. Попередня обробка даних

-Перетворити колонку з датою у формат datetime.

-Відсортувати дані за датою.

-Встановити дату як індекс часового ряду.

-Перевірити наявність пропущених значень та за необхідності обробити їх.

5. Дослідницький аналіз даних (EDA)

-Побудувати графік зміни температури у часі.

-Побудувати ковзне середнє значення температури.

-Побудувати гістограму розподілу температури.

-Побудувати boxplot для виявлення можливих викидів.

-Аналіз автокореляції

Побудувати графік автокореляційної функції (ACF).

Побудувати графік часткової автокореляції (PACF).

-Проаналізувати залежність значень часового ряду від попередніх періодів.

6.Підготовка даних для моделювання

-Розділити дані на тренувальну та тестову вибірки (наприклад, 80% / 20%).

7. Побудувати щонайменше три моделі прогнозування:

-ARIMA (класична статистична модель);

-Prophet (модель прогнозування часових рядів від Meta);

-LSTM (нейронна мережа для послідовних даних).

8. Оцінка моделей
Для кожної моделі обчислити метрики:

-MAE (Mean Absolute Error)

-RMSE (Root Mean Squared Error)

-MAPE (Mean Absolute Percentage Error)

8.Візуалізація результатів

-Побудувати графіки прогнозованих та фактичних значень.

-Побудувати порівняльний графік прогнозів усіх моделей.

-Порівняння моделей

-Створити таблицю з метриками точності.

9.Визначити модель, яка показала найкращий результат.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

In [ ]:
path = kagglehub.dataset_download("yug201/daily-climate-time-series-data-delhi-india")
print(os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "kaggel_weather_2013_to_2024.csv"))

print("Розмір:", df.shape)
display(df.head())
df.info()
display(df.describe())

Підготовка даних

In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%m/%d/%Y")
df = df.sort_values("DATE")

# Unnamed: 0 - це старий індекс, а Year, month, dayofweek, *_humidity, heat_index і т.д.
# пораховані з інших стовпців, тому залишаємо тільки основні погодні показники
keep = ["DATE", "temp", "tempmax", "tempmin", "humidity", "precip", "windspeed", "sealevelpressure"]
df = df[keep]

df = df.set_index("DATE")
df.head()

In [ ]:
df.info()
print()
print("Період:", df.index.min().date(), "-", df.index.max().date())

In [ ]:
print(df.isna().sum())
print()
print("Всього пропусків:", df.isna().sum().sum())
# пропусків немає, тому нічого заповнювати не треба

Графік температури

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(df.index, df["temp"], linewidth=0.8)
plt.title("Середня температура в Делі по днях")
plt.xlabel("Дата")
plt.ylabel("Температура, °C")
plt.show()

Ковзне середнє

In [ ]:
df["temp_ma30"] = df["temp"].rolling(window=30).mean()

plt.figure(figsize=(15, 5))
plt.plot(df.index, df["temp"], linewidth=0.6, alpha=0.5, label="Температура")
plt.plot(df.index, df["temp_ma30"], label="Ковзне середнє, 30 днів")
plt.title("Ковзне середнє температури")
plt.ylabel("Температура, °C")
plt.legend()
plt.show()

Розподіл температури

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["temp"], bins=40)
plt.title("Розподіл температури")
plt.xlabel("Температура, °C")
plt.ylabel("Кількість днів")
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=df["temp"])
plt.title("Boxplot температури")
plt.xlabel("Температура, °C")
plt.show()

Аналіз автокореляції
ACF

PACF

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_acf(df["temp"], lags=400, ax=ax)
plt.title("ACF температури")
plt.show()

Автокореляційна функція (ACF) використовується для аналізу залежності значень часового ряду від його попередніх значень. Вона відображає ступінь кореляції між значенням показника у певний момент часу та значеннями цього самого показника у попередні моменти часу, які називаються лагами. Побудова графіка автокореляційної функції дозволяє виявити, чи існує статистично значуща залежність між поточним значенням ряду та його попередніми спостереженнями. На такому графіку по горизонтальній осі відкладаються лаги, тобто кількість часових кроків назад, а по вертикальній — значення коефіцієнта автокореляції. Аналіз графіка ACF дає можливість виявити наявність тренду, сезонності або інших закономірностей у часовому ряді, а також використовується для визначення параметрів статистичних моделей прогнозування, зокрема моделі ARIMA.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_pacf(df["temp"], lags=40, ax=ax, method="ywm")
plt.title("PACF температури")
plt.show()

Часткова автокореляційна функція (PACF) застосовується для більш точного визначення залежності між значеннями часового ряду. На відміну від звичайної автокореляційної функції, PACF вимірює кореляцію між значенням ряду у певний момент часу та значенням у попередній момент часу з конкретним лагом, усуваючи вплив усіх проміжних лагів. Таким чином, часткова автокореляція показує «чисту» залежність між двома спостереженнями часового ряду. Побудова графіка PACF дозволяє визначити, які саме лаги мають статистично значущий вплив на значення ряду. Це є важливим етапом під час побудови моделей прогнозування часових рядів, оскільки аналіз графіка часткової автокореляції використовується для визначення порядку авторегресійної складової моделі, зокрема параметра
𝑝
p у моделі ARIMA.

**Аналіз:** на графіку ACF автокореляція на перших лагах майже 1 і спадає дуже повільно, потім стає від'ємною, а на лагах близько 350-400 знову додатна. Тобто температура сильно залежить від попередніх днів і є річна сезонність (літо і зима повторюються щороку).

На графіку PACF значущі в основному перші лаги, найсильніше лаг 1, далі значення маленькі. Тобто температура найбільше залежить від вчорашньої. Тому для ARIMA беремо невеликий p = 2, а d = 1, бо ряд не стаціонарний (є сезонні коливання).

In [ ]:
series = df["temp"]

# ділимо по часу: перші 80% - train, останні 20% - test (без перемішування)
split = int(len(series) * 0.8)
train = series.iloc[:split]
test = series.iloc[split:]

print("Train:", train.index.min().date(), "-", train.index.max().date(), "|", len(train), "записів")
print("Test:", test.index.min().date(), "-", test.index.max().date(), "|", len(test), "записів")

ARIMA

https://developer.ibm.com/tutorials/awb-arima-models-in-python/

In [ ]:
# p=2 за PACF (значущі перші лаги), d=1 бо ряд не стаціонарний, q=2
arima = ARIMA(train.values, order=(2, 1, 2))
arima_fit = arima.fit()

arima_pred = pd.Series(arima_fit.forecast(steps=len(test)), index=test.index)
arima_pred.head()

In [ ]:
print(arima_fit.summary())

сигма 2 : це представляє дисперсію залишкових значень, чим менше, тим краще

логарифмічна правдоподібність : це оцінка максимальної правдоподібності для моделі ARIMA, чим вище, тим краще

aic : міра ступеню відповідності та економності моделі, чим нижче, тим краще

In [ ]:
residuals = arima_fit.resid[1:]  # перший залишок пропускаємо, бо через d=1 він некоректний

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(residuals, linewidth=0.6)
axes[0].set_title("Залишки ARIMA")
axes[1].hist(residuals, bins=40)
axes[1].set_title("Розподіл залишків")
plot_acf(residuals, lags=40, ax=axes[2])
axes[2].set_title("ACF залишків")
plt.tight_layout()
plt.show()

print("Середнє залишків:", round(residuals.mean(), 4))
print("Стандартне відхилення залишків:", round(residuals.std(), 4))

**Аналіз залишків:** залишки коливаються навколо нуля (середнє майже 0), розподіл схожий на нормальний, але лівий хвіст трохи довший, тобто бувають дні з різким похолоданням, які модель не передбачає. На ACF залишків майже всі лаги в межах синьої зони, тому зв'язок з попередніми днями модель описала нормально.

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


def get_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE, %": float(mape(y_true, y_pred)),
    }


arima_metrics = get_metrics(test, arima_pred)
pd.Series(arima_metrics).round(3)

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(train.index[-365:], train[-365:], label="Train (останній рік)")
plt.plot(test.index, test, label="Факт (test)")
plt.plot(arima_pred.index, arima_pred, label="Прогноз ARIMA")
plt.title("ARIMA: факт і прогноз")
plt.ylabel("Температура, °C")
plt.legend()
plt.show()

PROPHET

https://facebook.github.io/prophet/docs/quick_start.html

In [ ]:
# Prophet потрібні стовпці ds (дата) і y (значення)
prophet_train = train.reset_index()
prophet_train.columns = ["ds", "y"]
prophet_train.head()

In [ ]:
prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
prophet.fit(prophet_train)

Прогноз

In [ ]:
future = pd.DataFrame({"ds": test.index})
forecast = prophet.predict(future)

prophet_pred = pd.Series(forecast["yhat"].values, index=test.index)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].head()

In [ ]:
prophet_metrics = get_metrics(test, prophet_pred)
pd.Series(prophet_metrics).round(3)

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(train.index[-365:], train[-365:], label="Train (останній рік)")
plt.plot(test.index, test, label="Факт (test)")
plt.plot(prophet_pred.index, prophet_pred, label="Прогноз Prophet")
plt.title("Prophet: факт і прогноз")
plt.ylabel("Температура, °C")
plt.legend()
plt.show()

In [ ]:
prophet.plot_components(forecast)
plt.show()

ПІДГОТОВКА ДАНИХ ДЛЯ LSTM

In [ ]:
window = 30  # прогнозуємо день за попередніми 30 днями

# scaler навчаємо тільки на train, щоб не підглядати в test
scaler = MinMaxScaler()
scaler.fit(train.values.reshape(-1, 1))
scaled = scaler.transform(series.values.reshape(-1, 1))

X_seq, y_seq, dates = [], [], []
for i in range(window, len(scaled)):
    X_seq.append(scaled[i - window:i, 0])
    y_seq.append(scaled[i, 0])
    dates.append(series.index[i])

X_seq = np.array(X_seq).reshape(-1, window, 1)
y_seq = np.array(y_seq)
dates = pd.DatetimeIndex(dates)
print("X:", X_seq.shape, "| y:", y_seq.shape)

In [ ]:
# у test потрапляють послідовності, де прогнозований день належить до test
is_test = dates >= test.index[0]

X_train_lstm, y_train_lstm = X_seq[~is_test], y_seq[~is_test]
X_test_lstm, y_test_lstm = X_seq[is_test], y_seq[is_test]

print("Train:", X_train_lstm.shape, "| Test:", X_test_lstm.shape)

In [ ]:
tf.keras.utils.set_random_seed(42)  # щоб результат був більш-менш однаковий при перезапуску

lstm = Sequential([
    Input(shape=(window, 1)),
    LSTM(64),
    Dense(1),
])
lstm.compile(optimizer="adam", loss="mse")

In [ ]:
history = lstm.fit(
    X_train_lstm, y_train_lstm,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    shuffle=False,
    verbose=2,  # один рядок на епоху
)

In [ ]:
lstm.summary()

In [ ]:
lstm_pred_scaled = lstm.predict(X_test_lstm, verbose=0)

# повертаємо значення в градуси
lstm_pred_values = scaler.inverse_transform(lstm_pred_scaled).flatten()
lstm_true_values = scaler.inverse_transform(y_test_lstm.reshape(-1, 1)).flatten()
print(lstm_pred_values[:5])
print(lstm_true_values[:5])

In [ ]:
lstm_dates = dates[is_test]
lstm_pred = pd.Series(lstm_pred_values, index=lstm_dates)
print("Перша дата:", lstm_dates[0].date(), "| остання:", lstm_dates[-1].date(), "| записів:", len(lstm_dates))

In [ ]:
lstm_metrics = get_metrics(test.loc[lstm_dates], lstm_pred)
pd.Series(lstm_metrics).round(3)

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(test.index, test, label="Факт (test)")
plt.plot(lstm_pred.index, lstm_pred, label="Прогноз LSTM")
plt.title("LSTM: факт і прогноз")
plt.ylabel("Температура, °C")
plt.legend()
plt.show()

Порівняння моделей

In [ ]:
results = pd.DataFrame([arima_metrics, prophet_metrics, lstm_metrics], index=["ARIMA", "Prophet", "LSTM"])
results = results.sort_values("MAE")
display(results.round(3))
print("Найкраща модель за MAE:", results.index[0])

plt.figure(figsize=(15, 6))
plt.plot(test.index, test, color="black", linewidth=1, label="Факт")
plt.plot(arima_pred.index, arima_pred, label="ARIMA")
plt.plot(prophet_pred.index, prophet_pred, label="Prophet")
plt.plot(lstm_pred.index, lstm_pred, label="LSTM")
plt.title("Порівняння прогнозів усіх моделей")
plt.ylabel("Температура, °C")
plt.legend()
plt.show()

## Висновок

Сформулюйте висновок за результатами роботи: коротко охарактеризуйте виконаний аналіз, порівняйте моделі ARIMA, Prophet та LSTM за метриками MAE, RMSE і MAPE та зазначте, яка модель продемонструвала найкращу якість прогнозування.

**Висновок:**

У роботі ми проаналізували щоденну температуру в Делі за 2013-2024 роки: побудували графік температури, ковзне середнє, гістограму, boxplot, ACF і PACF. Видно сильну річну сезонність: влітку температура доходить до 35-39 градусів, а взимку опускається до 10 і нижче.

Порівняння моделей на тестовій вибірці (з грудня 2022 по листопад 2024):
- ARIMA показала найгірший результат: MAE близько 10,9 °C, RMSE 12,5, MAPE майже 40%. Її прогноз на два роки вперед майже рівна лінія, бо проста ARIMA не враховує річну сезонність.
- Prophet значно краща: MAE близько 2 °C, RMSE 2,6, MAPE близько 9%. Вона добре вловила річну сезонність, але не повторює різкі зміни температури між днями.
- LSTM найкраща: MAE близько 1,1-1,4 °C, MAPE близько 5% (точні цифри трохи відрізняються при кожному запуску). Але треба враховувати, що LSTM прогнозує кожен день по реальних попередніх 30 днях, а ARIMA і Prophet прогнозують весь тестовий період одразу, тому в LSTM задача легша.

Отже, найкращу якість прогнозування показала модель LSTM, на другому місці Prophet, а найгірше ARIMA.